# Benchmarking tmol-mutscan against SKEMPI 2.0

**Does the predicted ΔΔG correlate with measured binding ΔΔG?**

Everything else in this project is mechanism-based reasoning. This notebook is
the experiment that replaces it with a number.

It takes measured binding affinity changes from
[SKEMPI 2.0](https://life.bsc.es/pid/skempi2/) — a curated database of
experimental ΔΔG values for mutations in protein–protein complexes — runs
`tmol-mutscan` on the same mutations, and reports how well the two agree.

---

### What this can and cannot settle

**Can settle:**

- Whether the predicted ranking of mutations carries real signal (Spearman ρ).
- The empirical REU → kcal/mol conversion, from the fitted slope.
- Whether the method is better at *destabilizing* than *stabilizing* mutations
  — the question that matters most, because peptide optimisation needs the
  stabilizing ones and that is where the method is theoretically weakest.
- Whether the v0.2.0 `strain` diagnostic improves accuracy when used as a
  filter.
- How many ensemble conformers are actually needed.

**Cannot settle:**

- Anything about peptide–protein complexes specifically. SKEMPI is
  protein–protein; a flexible peptide that is disordered when free is a harder
  case than anything here, so treat these numbers as an **upper bound** on
  performance for that application.
- Anything about protein–ligand complexes.
- Whether a specific predicted improvement is real. Only an experiment does
  that.

### Honest expectation

Published fixed-backbone Rosetta-style methods typically reach Pearson r in the
**0.4–0.6** range on SKEMPI-like sets, and do markedly worse on the stabilizing
subset. If this comes out in that band, the method is behaving as its class
does. Well below 0.3 would mean something is wrong with the setup, not just
with the physics.

Experimental replicates in SKEMPI agree to about **0.3 kcal/mol** (median SD
among repeated measurements), against a data spread of 1.9 kcal/mol — so
experimental noise caps Pearson r at roughly 0.99. The limit here is the
method, not the data.

### Sign convention

SKEMPI ΔΔG = RT·ln(K<sub>d,mut</sub> / K<sub>d,wt</sub>). **Positive = weaker
binding**, which matches `tmol-mutscan`'s convention, so the two are directly
comparable in sign.

In [ ]:
#@title 1. Check the runtime { display-mode: "form" }
#@markdown Turn the GPU on first: **Runtime → Change runtime type → T4 GPU**.
#@markdown A CPU-only benchmark of any useful size is not practical.

import shutil, subprocess, sys
from importlib.metadata import PackageNotFoundError, version

print("=" * 72)
HAS_GPU = False
if shutil.which("nvidia-smi"):
    probe = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    if probe.returncode == 0 and probe.stdout.strip():
        print("GPU           :", probe.stdout.strip())
        HAS_GPU = True
if not HAS_GPU:
    print("GPU           : NONE DETECTED")

major, minor = sys.version_info[:2]
print(f"Python        : {major}.{minor}")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = None
print(f"PyTorch       : {torch_version or 'not installed'}")

def _tuple(text):
    out = []
    for piece in text.split("+")[0].split("."):
        try: out.append(int(piece))
        except ValueError: break
    return tuple(out)

TORCH_NEEDS_UPGRADE = torch_version is None or _tuple(torch_version) < (2, 11)
print("-" * 72)
if (major, minor) < (3, 11):
    print("STOP: tmol needs Python 3.11+. Colab's version is not selectable.")
elif not HAS_GPU:
    print("NO GPU. Benchmarking on CPU is impractical beyond a couple of")
    print("complexes. Runtime -> Change runtime type -> T4 GPU, then re-run.")
else:
    print("Ready.")
if TORCH_NEEDS_UPGRADE:
    print("\nNOTE: PyTorch will be upgraded by the install; expect to restart")
    print("      the session afterwards. Cell 2 will tell you.")
print("=" * 72)

In [ ]:
#@title 2. Install tmol-mutscan v0.2.0 { display-mode: "form" }
#@markdown The benchmark needs v0.2.0 or newer: it uses `explicit_pairs`
#@markdown (to scan only the measured substitutions rather than the full cross
#@markdown product) and the `strain` diagnostic.
version_ref = "v0.2.0"  #@param {type:"string"}
cuda_build = "cu12"  #@param ["cu12", "cu13", "cpu"]

import subprocess, sys, time

REPO = "https://github.com/pablo-arantes/tmol-mutscan.git"
extra = "" if cuda_build == "cpu" else f"[{cuda_build}]"
requirement = f"tmol-mutscan{extra} @ git+{REPO}@{version_ref.strip() or 'main'}"
print("installing:", requirement, "\n")

started = time.perf_counter()
done = subprocess.run([sys.executable, "-m", "pip", "install", "-q", requirement],
                      capture_output=True, text=True)
tail = (done.stdout + done.stderr).strip().splitlines()
for line in tail[-12:]:
    print(line)
print(f"\npip exit {done.returncode} in {(time.perf_counter()-started)/60:.1f} min")
if done.returncode != 0:
    raise SystemExit("install failed; see output above")

changed_torch = any("torch" in l and ("Downloading" in l or "Installing" in l) for l in tail)
print("-" * 72)
if changed_torch or "torch" in sys.modules:
    print("RESTART THE SESSION NOW: Runtime -> Restart session, then go to cell 3.")
else:
    print("No restart needed. Continue to cell 3.")

In [ ]:
#@title 3. Verify { display-mode: "form" }
import torch, tmol, tmol_mutscan

print(f"tmol-mutscan : {tmol_mutscan.__version__}")
print(f"tmol         : {tmol.__version__}")
print(f"torch        : {torch.__version__}")
print(f"torch CUDA   : {torch.cuda.is_available()}")
print(f"tmol CUDA    : {tmol.has_cuda()}")

if tuple(int(x) for x in tmol_mutscan.__version__.split(".")[:2]) < (0, 2):
    raise SystemExit(
        f"this notebook needs tmol-mutscan >= 0.2.0, found "
        f"{tmol_mutscan.__version__}. Re-run cell 2 with version_ref = 'v0.2.0'."
    )

DEVICE = "cuda" if (torch.cuda.is_available() and tmol.has_cuda()) else "cpu"
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"\nGPU          : {p.name}, {p.total_memory/1024**3:.1f} GiB")
elif torch.cuda.is_available():
    print("\nPyTorch sees a GPU but TMol has no CUDA ops. Re-run cell 2 with")
    print("cuda_build='cu12' and restart. Falling back to CPU.")
print(f"\nDEVICE = {DEVICE!r}")

# Confirm the v0.2.0 API the benchmark depends on is actually present.
from tmol_mutscan.scan import build_requests_explicit, run_scan
import inspect
assert "explicit_pairs" in inspect.signature(run_scan).parameters
print("explicit_pairs API: present")

## 4. Get SKEMPI 2.0 and build the benchmark set

The parsing here encodes several things that are easy to get wrong, and each one
silently corrupts the benchmark if missed:

1. **Use `Mutation(s)_PDB`, not `Mutation(s)_cleaned`.** The cleaned column is
   renumbered. The same mutation is `LI45G` in PDB numbering and `LI38G`
   cleaned — and `tmol-mutscan` selections use PDB (author) numbering.
2. **Deduplicate.** SKEMPI stores one row per *measurement*, not per mutation.
   Some mutations appear six times from different papers, spanning more than a
   kcal/mol. Treating those as independent points would over-weight
   heavily-studied complexes and inflate the apparent sample size. We take the
   **median** per (complex, mutation).
3. **Stratify by complex.** Four serine protease–inhibitor complexes
   (1CHO, 1PPF, 1R0R, 3SGB) contribute 760 of the ~3,300 unique interface
   mutations. Without a per-complex cap, a quarter of the benchmark is one
   protein family.
4. **Interface only.** We keep `COR`/`SUP`/`RIM` and drop `SUR`/`INT`: a
   mutation that does not touch the interface has no interface energy to
   change, so it tests nothing here.

In [ ]:
#@title 4. Download and parse SKEMPI { display-mode: "form" }
import math, re
from pathlib import Path
import pandas as pd

WORKDIR = Path("/content/benchmark"); WORKDIR.mkdir(parents=True, exist_ok=True)
SKEMPI_CSV = WORKDIR / "skempi_v2.csv"
if not SKEMPI_CSV.is_file():
    import urllib.request
    url = "https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv"
    print("downloading", url)
    urllib.request.urlretrieve(url, SKEMPI_CSV)
print(f"{SKEMPI_CSV} ({SKEMPI_CSV.stat().st_size/1024:.0f} kB)")

raw = pd.read_csv(SKEMPI_CSV, sep=";", dtype=str)
print("rows:", len(raw), "| complexes:", raw["#Pdb"].nunique())

MUT_RE = re.compile(r"^([A-Z])([A-Za-z0-9])(-?\d+)([A-Za-z]?)([A-Z])$")
GAS_CONSTANT = 1.9872041e-3  # kcal/(mol K)

num = lambda s: pd.to_numeric(s, errors="coerce")
raw["kd_mut"] = num(raw["Affinity_mut_parsed"])
raw["kd_wt"] = num(raw["Affinity_wt_parsed"])
raw["temp_k"] = (
    raw["Temperature"].str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float).fillna(298.0)
)

keep = raw["Mutation(s)_PDB"].notna()
keep &= ~raw["Mutation(s)_PDB"].str.contains(",", na=True)     # single mutations
keep &= raw["Mutation(s)_PDB"].str.match(MUT_RE).fillna(False)
keep &= raw.kd_mut.gt(0) & raw.kd_wt.gt(0)
keep &= raw.temp_k.between(250, 350)
keep &= raw["iMutation_Location(s)"].isin(["COR", "SUP", "RIM"])

sel = raw[keep].copy()
sel["ddg_exp"] = GAS_CONSTANT * sel.temp_k * (sel.kd_mut / sel.kd_wt).apply(math.log)
print(f"usable single-point interface measurements: {len(sel)}")

# One row per unique (complex, mutation); median over replicate measurements.
grouped = sel.groupby(["#Pdb", "Mutation(s)_PDB"])["ddg_exp"]
SKEMPI = grouped.agg(ddg_exp="median", ddg_exp_spread="std", n_measurements="size")
SKEMPI = SKEMPI.reset_index().rename(columns={"#Pdb": "entry", "Mutation(s)_PDB": "mutation"})
SKEMPI["location"] = (
    sel.groupby(["#Pdb", "Mutation(s)_PDB"])["iMutation_Location(s)"].first().values
)
parsed = SKEMPI.mutation.str.extract(MUT_RE)
SKEMPI[["wt_aa", "chain", "resnum", "icode", "mut_aa"]] = parsed
SKEMPI["resnum"] = SKEMPI.resnum.astype(int)
SKEMPI["icode"] = SKEMPI.icode.fillna("")

print(f"unique (complex, mutation) pairs: {len(SKEMPI)}")
print(f"  destabilizing (>0): {int((SKEMPI.ddg_exp>0).sum())}")
print(f"  stabilizing   (<0): {int((SKEMPI.ddg_exp<0).sum())}")
replicated = SKEMPI[SKEMPI.n_measurements > 1]
print(f"  replicated pairs  : {len(replicated)}, "
      f"median spread {replicated.ddg_exp_spread.median():.2f} kcal/mol "
      "(the experimental noise floor)")

## 5. Structure preparation

Each SKEMPI entry names its partners in the `#Pdb` field: `1JTG_A_B` means
chain A against chain B, and `1CHO_EFG_I` means chains E, F, G against chain I.

Before loading, each PDB is filtered to **only the partner chains**, with
waters, glycans and common buffer/cryo components removed. This is not
cosmetic — it was arrived at by testing:

- Second copies of the complex in the asymmetric unit (1JTG has two) would
  otherwise sit in the pose and perturb the packing.
- **Glycans break loading outright.** TMol refuses to discard a sugar while
  keeping the Asn it is covalently bonded to, raising *"Cannot discard an
  incomplete or unsupported residue while retaining its covalent partner"*.
  Removing the sugar residues from the file fixes it. On a 28-complex probe
  this took the load success rate from 22/28 to 26/28.
- The two that still fail have **modified amino acids inside the chain**
  (CGU in 1DAN, IAS in 2FTL), which genuinely cannot be dropped without
  breaking the polymer. Those complexes are skipped and reported.

Metal ions are **kept** — they can be structural at an interface (1CSE has a
Ca²⁺) — and waters are dropped, which TMol does anyway.

Every mutation is then checked against the loaded structure: the wild-type
residue SKEMPI names must be the residue actually found at that chain and
number. On the probe this verified at essentially 100% for loaded complexes,
which is the evidence that the numbering convention is right. Any mutation that
fails is dropped and counted.

In [ ]:
#@title 5. Preparation and verification helpers { display-mode: "form" }
import re, urllib.error, urllib.request, warnings
from pathlib import Path

import torch
from tmol_mutscan.structure import load_structure, residue_table

PDB_CACHE = WORKDIR / "pdb"; PDB_CACHE.mkdir(exist_ok=True)
PREPARED = WORKDIR / "prepared"; PREPARED.mkdir(exist_ok=True)

# Waters, glycans and common crystallisation additives. Metals and genuine
# cofactors are deliberately NOT here.
DROP_RESIDUES = {
    "HOH", "DOD",                                              # water
    "NAG", "NDG", "BGC", "MAN", "BMA", "GAL", "FUC", "SIA",    # glycans
    "GLC", "XYS", "A2G", "GLA", "RAM",
    "GOL", "EDO", "PEG", "PG4", "SO4", "PO4", "ACT", "FMT",    # buffer / cryo
    "TRS", "MES", "DMS", "IPA", "MPD", "CAC", "IOD", "AZI",
}

def fetch_pdb(code: str) -> Path:
    path = PDB_CACHE / f"{code.lower()}.pdb"
    if not path.is_file():
        urllib.request.urlretrieve(
            f"https://files.rcsb.org/download/{code.upper()}.pdb", path
        )
    return path

def prepare_entry(entry: str) -> Path:
    """Filter a SKEMPI entry's PDB to its partner chains, cleaned."""
    code, chains_a, chains_b = entry.split("_")
    wanted = set(chains_a) | set(chains_b)
    out = PREPARED / f"{entry}.pdb"
    if out.is_file():
        return out
    kept = []
    for line in fetch_pdb(code).read_text().splitlines():
        tag = line[:6]
        if tag == "ENDMDL":
            break                                        # first model only
        if tag in ("ATOM  ", "HETATM"):
            if line[21] not in wanted:                      continue
            if line[17:20].strip() in DROP_RESIDUES:        continue
            if line[16] not in (" ", "A"):                  continue  # first altloc
            kept.append(line)
        elif tag == "SSBOND":
            kept.append(line)                            # disulfides matter
        # LINK records are dropped: they would reference removed glycans.
    if not kept:
        raise ValueError(f"no atoms left for {entry} after filtering")
    out.write_text("\n".join(kept) + "\nEND\n")
    return out

def load_entry(entry: str, device: torch.device):
    """Return (structure, residue lookup, partner selections) for an entry."""
    _, chains_a, chains_b = entry.split("_")
    path = prepare_entry(entry)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        structure = load_structure(str(path), device)
    table = residue_table(structure.pose)
    lookup = {(r.chain, r.resnum, r.icode): r for r in table}
    return structure, table, lookup, ",".join(chains_a), ",".join(chains_b)

def verify_mutations(lookup, rows):
    """Split SKEMPI rows into those whose wild-type identity checks out and not."""
    good, bad = [], []
    for row in rows.itertuples():
        residue = lookup.get((row.chain, int(row.resnum), row.icode or ""))
        if residue is None:
            bad.append((row.mutation, "position absent from structure"))
        elif residue.one_letter != row.wt_aa:
            bad.append((row.mutation, f"structure has {residue.one_letter}, SKEMPI says {row.wt_aa}"))
        else:
            good.append((row, residue))
    return good, bad

print("helpers defined")
print(f"  PDB cache     : {PDB_CACHE}")
print(f"  prepared files: {PREPARED}")

## 6. Choose the benchmark set

`max_mutations_per_complex` is the important one: without it the four protease
complexes dominate. The default samples mutations within each complex to span
its ΔΔG range, so a capped complex still contributes both stabilizing and
destabilizing cases rather than an arbitrary slice.

Cost scales as `complexes × conformers × (mutations + positions)` for the scan,
**plus** one ensemble relaxation per complex, which for a 400-residue complex is
usually the dominant term. Start small, look at the reported per-complex timing,
then raise `n_complexes`.

In [ ]:
#@title 6. Select the benchmark set { display-mode: "form" }
#@markdown ### Coverage
n_complexes = 25  #@param {type:"slider", min:2, max:120, step:1}
max_mutations_per_complex = 12  #@param {type:"slider", min:2, max:40, step:1}
#@markdown Skip complexes larger than this; relaxation cost grows steeply with size.
max_residues = 450  #@param {type:"slider", min:150, max:1200, step:50}
#@markdown Require at least this many usable mutations for a complex to be worth its ensemble.
min_mutations_per_complex = 4  #@param {type:"slider", min:1, max:20, step:1}
#@markdown Require at least this many *stabilizing* measurements, so the set can
#@markdown test the case the method is theoretically worst at. 0 disables.
min_stabilizing_per_complex = 1  #@param {type:"slider", min:0, max:5, step:1}

#@markdown ### Protocol
n_conformers = 3  #@param {type:"slider", min:1, max:15, step:1}
relax_repeats = 1  #@param {type:"slider", min:1, max:4, step:1}
minimize = True  #@param {type:"boolean"}
shell_gap = 5.0  #@param {type:"number"}
seed = 42  #@param {type:"integer"}

import numpy as np

pool = SKEMPI.copy()
by_complex = pool.groupby("entry")
eligible = []
for entry, rows in by_complex:
    if len(rows) < min_mutations_per_complex:
        continue
    if int((rows.ddg_exp < 0).sum()) < min_stabilizing_per_complex:
        continue
    eligible.append((entry, len(rows), int((rows.ddg_exp < 0).sum())))

eligible.sort(key=lambda t: -t[1])
print(f"complexes meeting the mutation criteria: {len(eligible)}")

def sample_rows(rows, cap, rng):
    """Take up to `cap` mutations spanning the complex's ddG range."""
    if len(rows) <= cap:
        return rows
    ordered = rows.sort_values("ddg_exp").reset_index(drop=True)
    picks = np.linspace(0, len(ordered) - 1, cap).round().astype(int)
    return ordered.iloc[sorted(set(picks))]

rng = np.random.default_rng(seed)
# Diversify: take complexes in descending mutation count but interleave so the
# set is not only the four protease systems.
chosen, selected_rows = [], []
for entry, _, _ in eligible:
    if len(chosen) >= n_complexes:
        break
    chosen.append(entry)
    selected_rows.append(sample_rows(pool[pool.entry == entry], max_mutations_per_complex, rng))

BENCH = pd.concat(selected_rows, ignore_index=True) if selected_rows else pd.DataFrame()
print(f"selected {BENCH.entry.nunique()} complexes, {len(BENCH)} mutations")
print(f"  destabilizing: {int((BENCH.ddg_exp>0).sum())}   "
      f"stabilizing: {int((BENCH.ddg_exp<0).sum())}")
print(f"  ddG_exp range: {BENCH.ddg_exp.min():.2f} to {BENCH.ddg_exp.max():.2f} kcal/mol")
print(f"  to-Ala share : {(BENCH.mut_aa=='A').mean()*100:.0f}%")
print()
print("mutations per selected complex:")
print(BENCH.entry.value_counts().to_string())
print()
print(f"NOTE: a complex is skipped if it has more than {max_residues} residues.")
print("Size is only known after loading, so skips appear during the run and are")
print("listed in the per-complex report at the end of cell 7. If everything gets")
print("skipped, raise max_residues: the protease complexes that dominate SKEMPI")
print("are 230-330 residues, and antibody complexes are 400-800.")

## 7. Run the benchmark

**Checkpointed and resumable.** Each complex's predictions are appended to
`predictions.csv` as soon as it finishes, and complexes already present are
skipped on a re-run. If Colab disconnects, just re-run this cell — nothing is
lost. Tick `save_to_drive` to keep the checkpoint outside the VM.

Run it once with `n_complexes = 2` in cell 6 first. That exercises the whole
pipeline in a few minutes and tells you the per-complex cost on the GPU you
were assigned.

In [ ]:
#@title 7. Run (resumable) { display-mode: "form" }
#@markdown Copy the checkpoint to Drive after each complex, so a disconnect
#@markdown cannot lose finished work.
save_to_drive = False  #@param {type:"boolean"}
drive_folder = "MyDrive/tmol-mutscan-benchmark"  #@param {type:"string"}
#@markdown Stop after this many minutes and keep what is done (0 = no limit).
time_budget_min = 0  #@param {type:"slider", min:0, max:360, step:10}

import json, shutil, time, traceback
from pathlib import Path

import pandas as pd
import torch

from tmol_mutscan.ensemble import generate_ensemble
from tmol_mutscan.scan import run_scan
from tmol_mutscan.stats import aggregate
from tmol_mutscan.logging_utils import setup_logging
from tmol.chemical import one2three
import logging

setup_logging()
logging.getLogger("tmol_mutscan").setLevel(logging.WARNING)   # keep output readable

PRED_CSV = WORKDIR / "predictions.csv"
STATUS_JSON = WORKDIR / "status.json"
device = torch.device(DEVICE)

done_entries, predictions = set(), []
if PRED_CSV.is_file():
    previous = pd.read_csv(PRED_CSV)
    predictions = previous.to_dict("records")
    done_entries = set(previous.entry.unique())
    print(f"resuming: {len(done_entries)} complexes, {len(previous)} predictions already done")

status = json.loads(STATUS_JSON.read_text()) if STATUS_JSON.is_file() else {}

def checkpoint():
    frame = pd.DataFrame(predictions)
    frame.to_csv(PRED_CSV, index=False)
    STATUS_JSON.write_text(json.dumps(status, indent=2))
    if save_to_drive:
        target = Path("/content/drive") / drive_folder.strip().lstrip("/")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(PRED_CSV, target / PRED_CSV.name)
        shutil.copy(STATUS_JSON, target / STATUS_JSON.name)
    return frame

if save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

entries = [e for e in BENCH.entry.unique() if e not in done_entries]
print(f"{len(entries)} complexes to run\n")
started_all = time.perf_counter()

for index, entry in enumerate(entries, 1):
    if time_budget_min and (time.perf_counter() - started_all) / 60 > time_budget_min:
        print(f"\ntime budget of {time_budget_min} min reached; stopping cleanly.")
        break

    rows = BENCH[BENCH.entry == entry]
    label = f"[{index}/{len(entries)}] {entry}"
    t0 = time.perf_counter()
    try:
        structure, table, lookup, partner_a, partner_b = load_entry(entry, device)
        n_res = len(table)
        if n_res > max_residues:
            status[entry] = {"status": "skipped", "reason": f"{n_res} residues > max_residues"}
            print(f"{label}: SKIP, {n_res} residues exceeds max_residues")
            checkpoint(); continue

        good, bad = verify_mutations(lookup, rows)
        if len(good) < min_mutations_per_complex:
            status[entry] = {"status": "skipped",
                             "reason": f"only {len(good)} of {len(rows)} mutations verified",
                             "failed": bad}
            print(f"{label}: SKIP, only {len(good)}/{len(rows)} mutations verified")
            checkpoint(); continue

        sfxn = structure.score_function()
        from tmol_mutscan.structure import parse_selection
        mask_a = parse_selection(structure.pose, partner_a, what="partner_a")
        mask_b = parse_selection(structure.pose, partner_b, what="partner_b")

        ensemble = generate_ensemble(
            structure.pose, sfxn, n_conformers=n_conformers, seed=seed,
            relax_repeats=relax_repeats, perturb=0.0,
        )
        expand = lambda m: m[0:1].expand(ensemble.n_poses, -1).contiguous()

        pairs = [(residue, one2three(row.mut_aa)) for row, residue in good]
        positions = sorted({residue for _, residue in good}, key=lambda r: r.block)
        raw, _ = run_scan(
            ensemble, positions, [], sfxn, structure.param_db,
            expand(mask_a), expand(mask_b),
            shell_gap=shell_gap, minimize=minimize, seed=seed,
            explicit_pairs=pairs,
        )
        stats = aggregate(raw)

        merged = 0
        for row, residue in good:
            hit = stats[(stats.position == residue.label)
                        & (stats.mut_aa == row.mut_aa)]
            if hit.empty:
                continue
            hit = hit.iloc[0]
            predictions.append({
                "entry": entry, "mutation": row.mutation, "location": row.location,
                "n_residues": n_res, "partner_a": partner_a, "partner_b": partner_b,
                "wt_aa": row.wt_aa, "mut_aa": row.mut_aa,
                "chain": row.chain, "resnum": int(row.resnum),
                "ddg_exp": float(row.ddg_exp),
                "n_measurements": int(row.n_measurements),
                "ddg_pred": float(hit.ddg_mean),
                "ddg_pred_sd": float(hit.ddg_sd),
                "ddg_pred_sem": float(hit.ddg_sem),
                "strain": float(hit.strain_mean),
                "ddg_total": float(hit.ddg_total_mean),
                "ddg_ref": float(hit.ddg_ref_mean),
                "strain_flag": bool(hit.strain_flag),
                "n_conformers": int(hit.n_conformers),
            })
            merged += 1

        elapsed = time.perf_counter() - t0
        status[entry] = {"status": "ok", "n_residues": n_res,
                         "mutations": merged, "unverified": bad,
                         "seconds": round(elapsed, 1)}
        print(f"{label}: {n_res} res, {merged} mutations in {elapsed/60:.1f} min"
              + (f", {len(bad)} unverified" if bad else ""))
    except Exception as exc:
        status[entry] = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"[:300]}
        print(f"{label}: FAILED {type(exc).__name__}: {str(exc)[:110]}")
        traceback.print_exc(limit=1)

    frame = checkpoint()
    if index % 5 == 0 or index == len(entries):
        print(f"      ... {len(frame)} predictions banked, "
              f"{(time.perf_counter()-started_all)/60:.1f} min elapsed")

RESULTS = checkpoint()
print(f"\n{'='*72}")
print(f"{len(RESULTS)} predictions over {RESULTS.entry.nunique() if len(RESULTS) else 0} complexes")
print(f"total time {(time.perf_counter()-started_all)/60:.1f} min")
ok = [e for e, s in status.items() if s.get("status") == "ok"]
print(f"complexes: {len(ok)} ok, "
      f"{sum(1 for s in status.values() if s.get('status')=='skipped')} skipped, "
      f"{sum(1 for s in status.values() if s.get('status')=='failed')} failed")
for entry, s in status.items():
    if s.get("status") != "ok":
        print(f"  {entry}: {s.get('status')} - {s.get('reason') or s.get('error')}")
print("=" * 72)

## 8. Results

Primary metrics are **Spearman ρ** and **Pearson r**, because they are
scale-free and REU are not kcal/mol. The fitted slope gives the empirical
REU → kcal/mol conversion; RMSE and MAE are reported only *after* applying it,
since raw RMSE between the two units is meaningless.

The breakdowns matter more than the headline number:

- **Stabilizing subset** (ΔΔG_exp < 0) — the case peptide optimisation needs,
  and the case this method is theoretically worst at.
- **Per-complex ρ** — a good pooled correlation can come entirely from getting
  the easy large-effect mutations right across different systems while ranking
  badly *within* each one. Within-complex ranking is what you actually use.
- **to-Ala vs other** — alanine scanning is the easy subset.

In [ ]:
#@title 8. Correlation and breakdowns { display-mode: "form" }
import numpy as np, pandas as pd
from scipy import stats as sps

_pred_path = WORKDIR / "predictions.csv"
R = pd.read_csv(_pred_path) if _pred_path.is_file() else pd.DataFrame()
if R.empty or "ddg_pred" not in R.columns:
    raise SystemExit(
        "No predictions were produced, so there is nothing to analyse.\n"
        "Check the per-complex report at the end of cell 7: every complex was\n"
        "skipped or failed. The usual causes are max_residues being too low for\n"
        "the complexes that were selected, or min_mutations_per_complex being\n"
        "higher than the number of mutations that verified. Loosen those in\n"
        "cell 6 and run cell 7 again."
    )
R = R[np.isfinite(R.ddg_exp) & np.isfinite(R.ddg_pred)].copy()
if len(R) < 4:
    raise SystemExit(
        f"Only {len(R)} prediction(s); correlations need more than that. "
        "Raise n_complexes in cell 6 and re-run cell 7 (it resumes, so "
        "nothing already computed is repeated)."
    )
print(f"{len(R)} predictions over {R.entry.nunique()} complexes\n")

def metrics(frame, name):
    if len(frame) < 4:
        return {"subset": name, "n": len(frame)}
    pear, p_pear = sps.pearsonr(frame.ddg_pred, frame.ddg_exp)
    spear, p_spear = sps.spearmanr(frame.ddg_pred, frame.ddg_exp)
    slope, intercept = np.polyfit(frame.ddg_pred, frame.ddg_exp, 1)
    scaled = slope * frame.ddg_pred + intercept
    resid = scaled - frame.ddg_exp
    both_signed = frame[(frame.ddg_exp.abs() > 0.5)]
    sign_ok = float((np.sign(both_signed.ddg_pred) == np.sign(both_signed.ddg_exp)).mean()) \
        if len(both_signed) else float("nan")
    return {
        "subset": name, "n": len(frame),
        "pearson_r": round(float(pear), 3), "p_pearson": f"{p_pear:.1e}",
        "spearman_rho": round(float(spear), 3), "p_spearman": f"{p_spear:.1e}",
        "slope_REU_to_kcal": round(float(slope), 3),
        "rmse_scaled": round(float(np.sqrt((resid**2).mean())), 3),
        "mae_scaled": round(float(resid.abs().mean()), 3),
        "sign_agreement": round(sign_ok, 3),
    }

rows = [
    metrics(R, "ALL"),
    metrics(R[R.ddg_exp > 0], "destabilizing (exp > 0)"),
    metrics(R[R.ddg_exp < 0], "STABILIZING (exp < 0)"),
    metrics(R[R.ddg_exp.abs() > 1], "|exp| > 1 kcal/mol"),
    metrics(R[R.mut_aa == "A"], "to-Ala"),
    metrics(R[R.mut_aa != "A"], "not to-Ala"),
    metrics(R[R.location == "COR"], "interface core"),
    metrics(R[R.location == "RIM"], "interface rim"),
]
SUMMARY = pd.DataFrame(rows)
pd.set_option("display.width", 220)
print(SUMMARY.to_string(index=False))

# Within-complex ranking: what you actually rely on in practice.
per_complex = []
for entry, g in R.groupby("entry"):
    if len(g) >= 5 and g.ddg_exp.nunique() > 2:
        rho, _ = sps.spearmanr(g.ddg_pred, g.ddg_exp)
        per_complex.append({"entry": entry, "n": len(g), "spearman_rho": round(float(rho), 3)})
# Explicit columns so the frame is still usable when no complex qualified.
PER_COMPLEX = pd.DataFrame(per_complex, columns=["entry", "n", "spearman_rho"])
if not PER_COMPLEX.empty:
    PER_COMPLEX = PER_COMPLEX.sort_values("spearman_rho", ascending=False)
    print(f"\nWITHIN-COMPLEX Spearman rho ({len(PER_COMPLEX)} complexes with >=5 mutations):")
    print(f"  median {PER_COMPLEX.spearman_rho.median():.3f}, "
          f"mean {PER_COMPLEX.spearman_rho.mean():.3f}, "
          f"{int((PER_COMPLEX.spearman_rho>0).sum())}/{len(PER_COMPLEX)} positive")
    print(PER_COMPLEX.to_string(index=False))

# Classification: can it pick out the badly destabilizing mutations?
hot = (R.ddg_exp > 1.0).to_numpy()
if hot.any() and (~hot).any():
    ranks = sps.rankdata(R.ddg_pred.to_numpy())
    n_pos, n_neg = int(hot.sum()), int((~hot).sum())
    auc = (ranks[hot].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
    print(f"\nAUC for detecting ddG_exp > 1 kcal/mol: {auc:.3f} "
          f"({n_pos} positive, {n_neg} negative; 0.5 = chance)")

In [ ]:
#@title 9. Figures { display-mode: "form" }
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats as sps

FIGDIR = WORKDIR / "figures"; FIGDIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))

# --- predicted vs measured -------------------------------------------------
ax = axes[0]
stab = R[R.ddg_exp < 0]
dest = R[R.ddg_exp >= 0]
ax.scatter(dest.ddg_pred, dest.ddg_exp, s=26, c="#2c7fb8", edgecolor="0.3",
           linewidth=0.3, label=f"destabilizing (n={len(dest)})", alpha=0.85)
ax.scatter(stab.ddg_pred, stab.ddg_exp, s=40, c="#d7301f", marker="^",
           edgecolor="0.2", linewidth=0.4, label=f"stabilizing (n={len(stab)})")
if len(R) >= 4:
    slope, intercept = np.polyfit(R.ddg_pred, R.ddg_exp, 1)
    xs = np.linspace(R.ddg_pred.min(), R.ddg_pred.max(), 50)
    ax.plot(xs, slope * xs + intercept, color="black", linewidth=1.3,
            label=f"fit: {slope:.2f}x + {intercept:.2f}")
    pear, _ = sps.pearsonr(R.ddg_pred, R.ddg_exp)
    spear, _ = sps.spearmanr(R.ddg_pred, R.ddg_exp)
    ax.set_title(f"Predicted vs measured   r = {pear:.3f}, rho = {spear:.3f}, n = {len(R)}")
ax.axhline(0, color="0.5", linewidth=0.8, linestyle="--")
ax.axvline(0, color="0.5", linewidth=0.8, linestyle="--")
ax.set_xlabel("predicted ddG (REU)")
ax.set_ylabel("measured ddG (kcal/mol)")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.3); ax.set_axisbelow(True)

# --- per-complex ranking ---------------------------------------------------
ax = axes[1]
if not PER_COMPLEX.empty:
    order = PER_COMPLEX.sort_values("spearman_rho")
    colors = ["#2c7fb8" if v > 0 else "#d7301f" for v in order.spearman_rho]
    ax.barh(range(len(order)), order.spearman_rho, color=colors)
    ax.set_yticks(range(len(order)),
                  [f"{e} (n={n})" for e, n in zip(order.entry, order.n)], fontsize=7)
    ax.axvline(0, color="black", linewidth=0.9)
    ax.axvline(order.spearman_rho.median(), color="black", linestyle=":", linewidth=1.1,
               label=f"median {order.spearman_rho.median():.2f}")
    ax.set_xlabel("within-complex Spearman rho")
    ax.set_title("Ranking quality inside each complex\n"
                 "this is what you rely on when choosing a mutation")
    ax.legend(frameon=False, fontsize=8)
    ax.grid(axis="x", alpha=0.3); ax.set_axisbelow(True)
else:
    ax.text(0.5, 0.5, "need >=5 mutations in a complex", ha="center", va="center")
    ax.axis("off")

fig.tight_layout()
fig.savefig(FIGDIR / "benchmark_overview.png", dpi=200, bbox_inches="tight")
plt.show()

# --- error vs magnitude, and the stabilizing subset alone ------------------
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
ax = axes2[0]
if len(R) >= 4:
    slope, intercept = np.polyfit(R.ddg_pred, R.ddg_exp, 1)
    resid = (slope * R.ddg_pred + intercept) - R.ddg_exp
    ax.scatter(R.ddg_exp, resid, s=24, c="#2c7fb8", edgecolor="0.3", linewidth=0.3)
    ax.axhline(0, color="black", linewidth=0.9)
    ax.set_xlabel("measured ddG (kcal/mol)")
    ax.set_ylabel("scaled prediction - measured (kcal/mol)")
    ax.set_title("Residuals: where the method goes wrong")
    ax.grid(alpha=0.3); ax.set_axisbelow(True)

ax = axes2[1]
if len(stab) >= 4:
    ax.scatter(stab.ddg_pred, stab.ddg_exp, s=42, c="#d7301f", marker="^",
               edgecolor="0.2", linewidth=0.4)
    rho, p = sps.spearmanr(stab.ddg_pred, stab.ddg_exp)
    ax.set_title(f"Stabilizing mutations only: rho = {rho:.3f} (p = {p:.2g}, n = {len(stab)})\n"
                 "the subset that matters for improving a binder")
    ax.axhline(0, color="0.5", linestyle="--", linewidth=0.8)
    ax.axvline(0, color="0.5", linestyle="--", linewidth=0.8)
    ax.set_xlabel("predicted ddG (REU)"); ax.set_ylabel("measured ddG (kcal/mol)")
    ax.grid(alpha=0.3); ax.set_axisbelow(True)
else:
    ax.text(0.5, 0.5, f"only {len(stab)} stabilizing mutations;\nraise n_complexes",
            ha="center", va="center"); ax.axis("off")

fig2.tight_layout()
fig2.savefig(FIGDIR / "benchmark_errors.png", dpi=200, bbox_inches="tight")
plt.show()
print("figures written to", FIGDIR)

## 10. Two things this benchmark can settle about the method

**The strain filter.** v0.2.0 reports `strain`, the part of the whole-pose
score change that is not cross-partner. The claim is that a mutation improving
the interface while straining the molecule is untrustworthy. If that claim is
right, dropping or correcting the flagged predictions should *improve* the
correlation. If the correlation does not move, the diagnostic is not earning
its place and should be treated as informational only.

**Ensemble size.** Recomputing the correlation using only the first *k*
conformers shows whether 3 was enough, or whether accuracy is still climbing
and production runs should use more. This is a far more meaningful convergence
test than the internal one, because it is measured against experiment rather
than against the protocol's own mean.

In [ ]:
#@title 10. Does the strain filter help? Does ensemble size matter? { display-mode: "form" }
import numpy as np
from scipy import stats as sps

print("=" * 72); print("STRAIN FILTER"); print("=" * 72)
if "strain" in R.columns and np.isfinite(R.strain).any():
    base_r, _ = sps.pearsonr(R.ddg_pred, R.ddg_exp)
    base_rho, _ = sps.spearmanr(R.ddg_pred, R.ddg_exp)
    print(f"baseline            : r = {base_r:.3f}, rho = {base_rho:.3f}, n = {len(R)}")

    flagged = R[R.strain_flag == True]  # noqa: E712
    print(f"flagged by strain   : {len(flagged)} of {len(R)}")
    kept = R[R.strain_flag != True]  # noqa: E712
    if len(kept) >= 4 and len(kept) < len(R):
        r2, _ = sps.pearsonr(kept.ddg_pred, kept.ddg_exp)
        rho2, _ = sps.spearmanr(kept.ddg_pred, kept.ddg_exp)
        print(f"excluding flagged   : r = {r2:.3f}, rho = {rho2:.3f}, n = {len(kept)}"
              f"   ({r2-base_r:+.3f} r)")

    # Using the whole-pose change instead of the interface change.
    if np.isfinite(R.ddg_total).any():
        r3, _ = sps.pearsonr(R.ddg_total, R.ddg_exp)
        rho3, _ = sps.spearmanr(R.ddg_total, R.ddg_exp)
        print(f"using ddg_total     : r = {r3:.3f}, rho = {rho3:.3f}"
              f"   ({r3-base_r:+.3f} r)")

    # A blend: interface energy penalised by its hidden strain.
    for weight in (0.25, 0.5, 1.0):
        blend = R.ddg_pred + weight * R.strain.fillna(0.0)
        rb, _ = sps.pearsonr(blend, R.ddg_exp)
        print(f"ddg + {weight:.2f}*strain   : r = {rb:.3f}   ({rb-base_r:+.3f} r)")
    print()
    print("Read it this way: if none of these beat the baseline by more than")
    print("about 0.02, the strain term is informational and should not be used")
    print("as a filter. If excluding flagged hits helps clearly, it should be.")
else:
    print("no strain data - was this run with tmol-mutscan >= 0.2.0?")

print()
print("=" * 72); print("ENSEMBLE SIZE"); print("=" * 72)
print("Needs the per-conformer table, which this notebook does not keep per")
print("complex. Instead: re-run the benchmark at a different n_conformers and")
print("compare the headline r. A cheap version is below, using the reported")
print("per-mutation SEM to estimate how much of the scatter is ensemble noise.")
if np.isfinite(R.ddg_pred_sem).any():
    noise = float(np.nanmean(R.ddg_pred_sem ** 2)) ** 0.5
    spread = float(R.ddg_pred.std())
    print(f"\nmean per-mutation SEM of the prediction : {noise:.3f} REU")
    print(f"SD of predictions across the benchmark   : {spread:.3f} REU")
    share = (noise / spread) ** 2 if spread > 0 else float("nan")
    print(f"=> ensemble noise accounts for ~{share*100:.1f}% of prediction variance")
    if np.isfinite(share) and share < 0.05:
        print("   Small: more conformers would barely move the correlation.")
        print("   The error is in the physics, not the sampling.")
    else:
        print("   Large enough to matter: try raising n_conformers and re-running.")

In [ ]:
#@title 11. Export everything { display-mode: "form" }
#@markdown Download a zip with the predictions, metrics and figures.
download = True  #@param {type:"boolean"}
also_drive = False  #@param {type:"boolean"}

import json, shutil, time
from pathlib import Path
import pandas as pd

SUMMARY.to_csv(WORKDIR / "metrics_summary.csv", index=False)
if not PER_COMPLEX.empty:
    PER_COMPLEX.to_csv(WORKDIR / "metrics_per_complex.csv", index=False)

provenance = {
    "tmol_mutscan": tmol_mutscan.__version__,
    "tmol": tmol.__version__,
    "torch": torch.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "n_complexes_requested": int(n_complexes),
    "max_mutations_per_complex": int(max_mutations_per_complex),
    "max_residues": int(max_residues),
    "n_conformers": int(n_conformers),
    "relax_repeats": int(relax_repeats),
    "minimize": bool(minimize),
    "shell_gap": float(shell_gap),
    "seed": int(seed),
    "n_predictions": int(len(R)),
    "n_complexes_done": int(R.entry.nunique()),
    "skempi_source": "https://life.bsc.es/pid/skempi2/database/download/skempi_v2.csv",
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
(WORKDIR / "benchmark_manifest.json").write_text(json.dumps(provenance, indent=2))

stamp = time.strftime("%Y%m%d-%H%M%S")
archive = shutil.make_archive(f"/content/skempi_benchmark_{stamp}", "zip", root_dir=str(WORKDIR),
                              base_dir=".")
print(f"archive: {archive} ({Path(archive).stat().st_size/1024**2:.1f} MB)")

if also_drive:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    target = Path("/content/drive") / drive_folder.strip().lstrip("/")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, target / Path(archive).name)
    print("copied to Drive:", target / Path(archive).name)

if download:
    from google.colab import files
    files.download(archive)

print()
print("=" * 72)
print("PASTE THIS BACK (the headline result)")
print("=" * 72)
print(json.dumps(provenance, indent=2))
print()
print(SUMMARY.to_string(index=False))
if not PER_COMPLEX.empty:
    print(f"\nwithin-complex rho: median {PER_COMPLEX.spearman_rho.median():.3f} "
          f"over {len(PER_COMPLEX)} complexes, "
          f"{int((PER_COMPLEX.spearman_rho>0).sum())} positive")
print("=" * 72)
print("\nThe zip also contains predictions.csv (every mutation, predicted and")
print("measured), status.json (what was skipped and why), and the figures.")

---

## Interpreting the result

**Pearson r / Spearman ρ on ALL** — the headline. For reference, fixed-backbone
Rosetta-class methods on SKEMPI-like sets usually land around 0.4–0.6.

**The stabilizing subset is the real test for peptide optimisation.** If ρ there
is near zero or negative while the overall ρ looks respectable, then the method
ranks damaging mutations well and improving mutations not at all — which means
it should be used to *exclude* candidates, never to *rank* them. That is the
single most decision-relevant number in this notebook.

**`slope_REU_to_kcal`** is the empirical unit conversion. If it comes out near
1.0, REU happen to track kcal/mol on this set; far from 1.0 means predicted
magnitudes should not be quoted in energy units at all.

**Within-complex ρ** is what you actually use. A good pooled r with a median
within-complex ρ near zero means the method separates systems, not mutations —
useless for choosing which residue to change in *your* complex.

**`sign_agreement`** on mutations with |ΔΔG_exp| > 0.5: the fraction where the
predicted direction is right. Below ~0.7 and the predictions are close to
coin-flips on direction.

### Caveats to carry into any write-up

- SKEMPI is **protein–protein**. These numbers are an upper bound for
  peptide–protein, where the free-state peptide ensemble matters and is not
  modelled, and say nothing about protein–ligand.
- `max_residues` biases the set toward **smaller complexes**.
- Glycosylated entries and those with modified residues in-chain are **excluded**
  (see `status.json`), which removes much of the antibody–antigen set.
- `n_conformers = 3` gives noisy per-mutation estimates, so the correlation is
  a **lower bound** on what the protocol can do. Cell 10 estimates how much
  that costs.
- Experimental ΔΔG values are medians over replicates agreeing to ~0.3
  kcal/mol, which caps attainable r near 0.99 — not the limiting factor.

### Reproducing

`benchmark_manifest.json` records every setting, the library versions and the
GPU. The benchmark is deterministic given the same seed, versions and device.